# 🎓 Diploma Thesis: Automatic Detection of Toxic Content
## Datasets: RU Paradetox (Russian) + Jigsaw Toxic Comments (English)
## Pipeline: EDA → Preprocessing → Baseline ML → ruBERT Fine-tuning → SHAP

---
## ⚠️ ПЕРЕД ЗАПУСКОМ — ОБЯЗАТЕЛЬНО:
### Добавь датасет Jigsaw на Kaggle:
1. Справа нажми **+ Add Data**
2. Найди: `jigsaw-toxic-comment-classification-challenge`
3. Нажми **Add** → датасет появится в `/kaggle/input/`
4. Включи **GPU T4** в Session Options → Accelerator
5. Нажми **Run All**
---

## Step 1: Install Libraries

In [ ]:
!pip install -q datasets transformers torch scikit-learn pandas numpy matplotlib seaborn nltk wordcloud tqdm shap

## Step 2: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import re
import os
import warnings
import nltk

from wordcloud import WordCloud
from datasets import load_dataset
from tqdm.notebook import tqdm

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, classification_report, confusion_matrix, roc_curve
)

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
import shap

warnings.filterwarnings('ignore')
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 13
sns.set_style('whitegrid')
tqdm.pandas()

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Libraries loaded')
print(f'🖥️  Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  GPU not found! Go to Session Options → Accelerator → GPU T4')

## Step 3: Load Datasets

In [ ]:
# ── 3a: Russian dataset — RU Paradetox (auto-downloaded from HuggingFace) ──
print('📥 Loading Russian dataset (ru_paradetox)...')
dataset = load_dataset('s-nlp/ru_paradetox')
df_ru = pd.DataFrame(dataset['train'])
print(f'   Columns: {df_ru.columns.tolist()}')

toxic_ru   = pd.DataFrame({'text': df_ru['ru_toxic_comment'],   'label': 1, 'label_name': 'toxic',     'lang': 'ru'})
neutral_ru = pd.DataFrame({'text': df_ru['ru_neutral_comment'], 'label': 0, 'label_name': 'non-toxic', 'lang': 'ru'})
df_ru_final = pd.concat([toxic_ru, neutral_ru], ignore_index=True).dropna(subset=['text'])

print(f'✅ Russian dataset: {len(df_ru_final)} examples')
print(df_ru_final['label_name'].value_counts())

In [ ]:
# ── 3b: English dataset — Jigsaw Toxic Comments ────────────────────────────
# Kaggle path (after adding dataset via + Add Data)
JIGSAW_PATH = '/kaggle/input/jigsaw-toxic-comment-classification-challenge/train.csv'

if not os.path.exists(JIGSAW_PATH):
    raise FileNotFoundError(
        '❌ Jigsaw dataset not found!\n'
        'Please add it via: + Add Data → search jigsaw-toxic-comment-classification-challenge → Add'
    )

print(f'📥 Loading Jigsaw from: {JIGSAW_PATH}')
df_jigsaw_raw = pd.read_csv(JIGSAW_PATH)
print(f'   Raw size: {df_jigsaw_raw.shape}')
print(f'   Columns: {df_jigsaw_raw.columns.tolist()}')

# Binary label: if any toxic column == 1 → toxic
toxic_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
df_jigsaw_raw['label'] = (df_jigsaw_raw[toxic_cols].sum(axis=1) > 0).astype(int)

print(f'\nJigsaw class distribution (before balancing):')
print(df_jigsaw_raw['label'].value_counts())

# Balance: 8000 toxic + 8000 non-toxic = 16000 EN examples
N_EN = 8000
n_toxic_available = df_jigsaw_raw['label'].sum()
jigsaw_toxic   = df_jigsaw_raw[df_jigsaw_raw['label']==1].sample(n=min(N_EN, n_toxic_available), random_state=RANDOM_SEED)
jigsaw_neutral = df_jigsaw_raw[df_jigsaw_raw['label']==0].sample(n=N_EN, random_state=RANDOM_SEED)

df_en_final = pd.concat([
    pd.DataFrame({'text': jigsaw_toxic['comment_text'],   'label': 1, 'label_name': 'toxic',     'lang': 'en'}),
    pd.DataFrame({'text': jigsaw_neutral['comment_text'], 'label': 0, 'label_name': 'non-toxic', 'lang': 'en'}),
], ignore_index=True).dropna(subset=['text'])

print(f'\n✅ English dataset (balanced): {len(df_en_final)} examples')
print(df_en_final['label_name'].value_counts())

In [ ]:
# ── 3c: Merge RU + EN ──────────────────────────────────────────────────────
df_final = pd.concat([df_ru_final, df_en_final], ignore_index=True)
df_final = df_final.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print('=' * 60)
print(f'📊 COMBINED DATASET: {len(df_final)} examples')
print(f'   Russian (RU):  {len(df_ru_final)}')
print(f'   English (EN):  {len(df_en_final)}')
print(f'\nClass distribution:')
print(df_final['label_name'].value_counts())
print(f'\nBy language:')
print(df_final.groupby(['lang', 'label_name']).size())
print('=' * 60)
df_final.head(5)

## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# Chart 1: Class + Language distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#e74c3c', '#2ecc71']

counts = df_final['label_name'].value_counts()
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='black', width=0.5)
axes[0].set_title('Class Distribution (Overall)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 80, str(v), ha='center', fontweight='bold')

lang_counts = df_final['lang'].value_counts()
axes[1].bar(['Russian (RU)', 'English (EN)'], lang_counts.values,
            color=['#3498db', '#e67e22'], edgecolor='black', width=0.5)
axes[1].set_title('Distribution by Language', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Count')
for i, v in enumerate(lang_counts.values):
    axes[1].text(i, v + 80, str(v), ha='center', fontweight='bold')

axes[2].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize': 13})
axes[2].set_title('Class Share (%)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 1 saved: class_distribution.png')

In [ ]:
# Chart 2: Text length distribution
df_final['text_len']   = df_final['text'].apply(lambda x: len(str(x)))
df_final['word_count'] = df_final['text'].apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for label, color in [('toxic', '#e74c3c'), ('non-toxic', '#2ecc71')]:
    subset = df_final[df_final['label_name'] == label]
    axes[0][0].hist(subset['text_len'].clip(upper=500),  bins=40, alpha=0.6, label=label, color=color)
    axes[0][1].hist(subset['word_count'].clip(upper=100), bins=40, alpha=0.6, label=label, color=color)

for lang, color in [('ru', '#3498db'), ('en', '#e67e22')]:
    subset = df_final[df_final['lang'] == lang]
    axes[1][0].hist(subset['text_len'].clip(upper=500),  bins=40, alpha=0.6, label=lang.upper(), color=color)
    axes[1][1].hist(subset['word_count'].clip(upper=100), bins=40, alpha=0.6, label=lang.upper(), color=color)

titles = ['Text length (chars) — by class', 'Text length (words) — by class',
          'Text length (chars) — by language', 'Text length (words) — by language']
for ax, title in zip(axes.flat, titles):
    ax.set_title(title, fontweight='bold')
    ax.legend()

plt.tight_layout()
plt.savefig('text_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 2 saved: text_length_distribution.png')

In [ ]:
# Chart 3: WordCloud — 4 clouds (RU toxic / RU neutral / EN toxic / EN neutral)
RUSSIAN_STOP = set(stopwords.words('russian'))
ENGLISH_STOP = set(stopwords.words('english'))

def get_words(df_subset, lang):
    all_text = ' '.join(df_subset['text'].astype(str).tolist())
    all_text = re.sub(r'[^а-яёА-ЯЁa-zA-Z\s]', '', all_text)
    stop = RUSSIAN_STOP if lang == 'ru' else ENGLISH_STOP
    words = [w.lower() for w in all_text.split() if w.lower() not in stop and len(w) > 2]
    return ' '.join(words)

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
configs = [
    (0, 0, 'ru', 1, '☠️ Russian — Toxic',      '#2c2c2c', 'Reds'),
    (0, 1, 'ru', 0, '✅ Russian — Non-toxic',   'white',   'Greens'),
    (1, 0, 'en', 1, '☠️ English — Toxic',       '#2c2c2c', 'OrRd'),
    (1, 1, 'en', 0, '✅ English — Non-toxic',   'white',   'Blues'),
]
for r, c, lang, lbl, title, bg, cmap in configs:
    subset = df_final[(df_final['lang'] == lang) & (df_final['label'] == lbl)]
    words  = get_words(subset, lang)
    if len(words) > 10:
        wc = WordCloud(width=700, height=400, background_color=bg,
                       colormap=cmap, max_words=80).generate(words)
        axes[r][c].imshow(wc, interpolation='bilinear')
    axes[r][c].set_title(title, fontsize=14, fontweight='bold')
    axes[r][c].axis('off')

plt.tight_layout()
plt.savefig('wordcloud.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 3 saved: wordcloud.png')

## Step 5: Text Preprocessing (NLP Pipeline)

In [ ]:
RUSSIAN_STOP = set(stopwords.words('russian'))
ENGLISH_STOP = set(stopwords.words('english'))
ALL_STOP     = RUSSIAN_STOP | ENGLISH_STOP

def clean_text(text):
    """Universal cleaning for RU + EN texts"""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)        # remove URLs
    text = re.sub(r'@\w+|#\w+', '', text)              # remove mentions/hashtags
    text = re.sub(r'[^а-яёА-ЯЁa-zA-Z\s]', ' ', text)  # letters only
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def remove_stopwords(text):
    words = text.split()
    return ' '.join(w for w in words if w not in ALL_STOP and len(w) > 2)

print('⚙️ Applying NLP pipeline...')
df_final['text_clean'] = df_final['text'].progress_apply(clean_text)
df_final['text_lemma'] = df_final['text_clean'].progress_apply(remove_stopwords)
df_final = df_final[df_final['text_lemma'].str.strip().str.len() > 0].reset_index(drop=True)

print(f'✅ Done! Examples after cleaning: {len(df_final)}')
df_final[['text', 'text_clean', 'text_lemma', 'label', 'lang']].head(5)

## Step 6: Train / Val / Test Split

In [ ]:
X      = df_final['text_lemma'].astype(str)   # for TF-IDF
X_bert = df_final['text_clean'].astype(str)   # for BERT
y      = df_final['label']

# 70% train / 15% val / 15% test — stratified
X_train, X_temp, X_bert_train, X_bert_temp, y_train, y_temp = train_test_split(
    X, X_bert, y, test_size=0.30, random_state=RANDOM_SEED, stratify=y
)
X_val, X_test, X_bert_val, X_bert_test, y_val, y_test = train_test_split(
    X_temp, X_bert_temp, y_temp, test_size=0.50, random_state=RANDOM_SEED, stratify=y_temp
)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')
print(f'Train toxic ratio: {y_train.mean():.3f} | Test toxic ratio: {y_test.mean():.3f}')

## Step 7: Baseline Models (TF-IDF + ML)

In [ ]:
# TF-IDF supports both Cyrillic and Latin simultaneously
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=80000,
    sublinear_tf=True,
    min_df=2,
    analyzer='word'
)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)
print(f'TF-IDF matrix shape: {X_train_tfidf.shape}')

def evaluate_model(name, model, needs_calibration=False):
    print(f'\n🔄 Training: {name}...')
    if needs_calibration:
        model = CalibratedClassifierCV(model, cv=3)
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    y_prob = model.predict_proba(X_test_tfidf)[:, 1]
    metrics = {
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall':    round(recall_score(y_test, y_pred), 4),
        'F1-Score':  round(f1_score(y_test, y_pred), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, y_prob), 4),
    }
    print(f'   F1: {metrics["F1-Score"]} | AUC: {metrics["ROC-AUC"]}')
    return metrics, model, y_pred, y_prob

all_metrics, trained_models, all_preds, all_probs = [], {}, {}, {}

for name, model, cal in [
    ('Logistic Regression', LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_SEED), False),
    ('SVM (LinearSVC)',     LinearSVC(C=1.0, max_iter=2000, random_state=RANDOM_SEED),          True),
    ('Random Forest',       RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED), False),
]:
    m, tm, yp, ypr = evaluate_model(name, model, cal)
    all_metrics.append(m)
    trained_models[name] = tm
    all_preds[name] = yp
    all_probs[name] = ypr

baseline_df = pd.DataFrame(all_metrics)
print('\n📊 Baseline results:')
print(baseline_df.to_string(index=False))

In [ ]:
# Chart 4: Baseline model comparison
metrics_cols = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metrics_cols))
width = 0.25
colors3 = ['#3498db', '#e74c3c', '#2ecc71']

fig, ax = plt.subplots(figsize=(14, 6))
for i, row in baseline_df.iterrows():
    vals = [row[m] for m in metrics_cols]
    bars = ax.bar(x + i*width, vals, width, label=row['Model'], color=colors3[i], alpha=0.85, edgecolor='black')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_cols)
ax.set_ylim(0.7, 1.04)
ax.set_title('Baseline Model Comparison (RU + EN Dataset)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('baseline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 4 saved')

In [ ]:
# Chart 5: Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, y_pred) in zip(axes, all_preds.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Non-toxic','Toxic'],
                yticklabels=['Non-toxic','Toxic'],
                annot_kws={'fontsize': 14})
    ax.set_title(f'Confusion Matrix\n{name}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('confusion_matrices_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 5 saved')

In [ ]:
# Chart 6: ROC Curves
fig, ax = plt.subplots(figsize=(9, 7))
for (name, y_prob), color in zip(all_probs.items(), colors3):
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, lw=2.5, color=color, label=f'{name} (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'k--', lw=1.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Baseline Models (RU + EN)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 6 saved')

## Step 8: ruBERT Fine-tuning
### Note: ruBERT (DeepPavlov) works well on both Russian and English texts.
### It covers Cyrillic + Latin tokens — optimal for mixed RU+EN dataset.

In [ ]:
MODEL_NAME = 'DeepPavlov/rubert-base-cased'
MAX_LEN    = 128
BATCH_SIZE = 32
EPOCHS     = 3
LR         = 2e-5

print(f'📥 Loading tokenizer: {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('✅ Tokenizer loaded')

class ToxicDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts  = texts.tolist() if hasattr(texts, 'tolist') else list(texts)
        self.labels = labels.tolist() if hasattr(labels, 'tolist') else list(labels)
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx], max_length=MAX_LEN,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_loader = DataLoader(ToxicDataset(X_bert_train, y_train), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(ToxicDataset(X_bert_val,   y_val),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(ToxicDataset(X_bert_test,  y_test),  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

In [ ]:
print('📥 Loading ruBERT model...')
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)

total_steps = len(train_loader) * EPOCHS
optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler   = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps
)
scaler = GradScaler()

print(f'✅ Model loaded | Parameters: {sum(p.numel() for p in model.parameters()):,}')

def train_epoch(model, loader):
    model.train()
    total_loss, preds_all, labels_all = 0, [], []
    for batch in tqdm(loader, desc='Training', leave=False):
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        labs = batch['label'].to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            out = model(input_ids=ids, attention_mask=mask, labels=labs)
        scaler.scale(out.loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += out.loss.item()
        preds_all.extend(torch.argmax(out.logits, dim=1).cpu().numpy())
        labels_all.extend(labs.cpu().numpy())
    return total_loss / len(loader), f1_score(labels_all, preds_all)

def eval_epoch(model, loader):
    model.eval()
    total_loss, preds_all, labels_all, probs_all = 0, [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='Evaluating', leave=False):
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            labs = batch['label'].to(DEVICE)
            with autocast():
                out = model(input_ids=ids, attention_mask=mask, labels=labs)
            total_loss += out.loss.item()
            probs_all.extend(torch.softmax(out.logits, dim=1)[:, 1].cpu().numpy())
            preds_all.extend(torch.argmax(out.logits, dim=1).cpu().numpy())
            labels_all.extend(labs.cpu().numpy())
    return (
        total_loss / len(loader),
        f1_score(labels_all, preds_all),
        roc_auc_score(labels_all, probs_all),
        preds_all, probs_all, labels_all
    )

print('✅ Training functions ready')

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': [], 'val_auc': []}
best_f1   = 0
SAVE_PATH = 'rubert_best.pt'

print(f'🚀 Training ruBERT on RU+EN dataset ({EPOCHS} epochs)...')
for epoch in range(1, EPOCHS + 1):
    print(f'\n📌 Epoch {epoch}/{EPOCHS}')
    tr_loss, tr_f1 = train_epoch(model, train_loader)
    vl_loss, vl_f1, vl_auc, _, _, _ = eval_epoch(model, val_loader)
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_f1'].append(tr_f1)
    history['val_f1'].append(vl_f1)
    history['val_auc'].append(vl_auc)
    print(f'   Train → Loss:{tr_loss:.4f}  F1:{tr_f1:.4f}')
    print(f'   Val   → Loss:{vl_loss:.4f}  F1:{vl_f1:.4f}  AUC:{vl_auc:.4f}')
    if vl_f1 > best_f1:
        best_f1 = vl_f1
        torch.save(model.state_dict(), SAVE_PATH)
        print(f'   💾 Best model saved (F1={best_f1:.4f})')

print(f'\n✅ Training complete! Best Val F1: {best_f1:.4f}')

In [ ]:
# Chart 7: Training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, EPOCHS + 1)

axes[0].plot(ep, history['train_loss'], 'o-', color='#e74c3c', lw=2.5, label='Train Loss')
axes[0].plot(ep, history['val_loss'],   's-', color='#3498db', lw=2.5, label='Val Loss')
axes[0].set_title('Loss per Epoch (ruBERT, RU+EN)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].set_xticks(list(ep))

axes[1].plot(ep, history['train_f1'], 'o-', color='#e74c3c', lw=2.5, label='Train F1')
axes[1].plot(ep, history['val_f1'],   's-', color='#3498db', lw=2.5, label='Val F1')
axes[1].plot(ep, history['val_auc'],  '^-', color='#2ecc71', lw=2.5, label='Val AUC')
axes[1].set_title('F1 and AUC per Epoch (ruBERT, RU+EN)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].set_xticks(list(ep))

plt.tight_layout()
plt.savefig('bert_training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 7 saved')

In [ ]:
# Evaluate BERT on test set
model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
_, test_f1, test_auc, test_preds, test_probs, test_true = eval_epoch(model, test_loader)

bert_metrics = {
    'Model':     'ruBERT (fine-tuned)',
    'Accuracy':  round(accuracy_score(test_true, test_preds), 4),
    'Precision': round(precision_score(test_true, test_preds), 4),
    'Recall':    round(recall_score(test_true, test_preds), 4),
    'F1-Score':  round(test_f1, 4),
    'ROC-AUC':   round(test_auc, 4),
}
print('🏆 ruBERT results (RU+EN test set):')
for k, v in bert_metrics.items():
    print(f'   {k}: {v}')
print('\n', classification_report(test_true, test_preds, target_names=['Non-toxic', 'Toxic']))

In [ ]:
# Chart 8: BERT Confusion Matrix + ROC
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(test_true, test_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', ax=axes[0],
            xticklabels=['Non-toxic','Toxic'], yticklabels=['Non-toxic','Toxic'],
            annot_kws={'fontsize': 16})
axes[0].set_title('Confusion Matrix — ruBERT (RU+EN)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

fpr, tpr, _ = roc_curve(test_true, test_probs)
axes[1].plot(fpr, tpr, lw=2.5, color='#e67e22', label=f'ruBERT (AUC={test_auc:.4f})')
axes[1].plot([0,1],[0,1],'k--', lw=1.5)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — ruBERT (RU+EN)', fontsize=13, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('bert_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 8 saved')

## Step 9: Final Model Comparison

In [ ]:
all_results = pd.concat([baseline_df, pd.DataFrame([bert_metrics])], ignore_index=True)

print('📊 FINAL COMPARISON TABLE (RU + EN dataset)')
print('=' * 80)
print(all_results.to_string(index=False))
print('=' * 80)
best = all_results.loc[all_results['F1-Score'].idxmax(), 'Model']
print(f'\n🏆 Best model: {best}')

all_results.to_csv('final_results.csv', index=False)
print('✅ Table saved: final_results.csv')

In [ ]:
# Chart 9: All models comparison
metrics_cols = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metrics_cols))
width = 0.2
colors4 = ['#3498db', '#e74c3c', '#2ecc71', '#e67e22']

fig, ax = plt.subplots(figsize=(16, 7))
for i, row in all_results.iterrows():
    vals   = [row[m] for m in metrics_cols]
    offset = (i - len(all_results)/2 + 0.5) * width
    bars   = ax.bar(x + offset, vals, width, label=row['Model'],
                    color=colors4[i], alpha=0.85, edgecolor='black')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                f'{val:.3f}', ha='center', fontsize=8.5, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(metrics_cols, fontsize=13)
ax.set_ylim(0.70, 1.05)
ax.set_title('Comparative Analysis of All Models (RU + EN Dataset)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.35)
plt.tight_layout()
plt.savefig('final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 9 saved')

In [ ]:
# Chart 10: Radar chart
categories = ['Accuracy','Precision','Recall','F1-Score','ROC-AUC']
N = len(categories)
angles = [n/float(N)*2*np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for i, row in all_results.iterrows():
    vals = [row[c] for c in categories] + [row[categories[0]]]
    ax.plot(angles, vals, lw=2.5, color=colors4[i], label=row['Model'])
    ax.fill(angles, vals, alpha=0.08, color=colors4[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=12)
ax.set_ylim(0.7, 1.0)
ax.set_title('Radar Chart — Model Metrics (RU + EN)', size=15, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10)
plt.tight_layout()
plt.savefig('radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 10 saved')

## Step 10: SHAP — Model Interpretability

In [ ]:
# SHAP for Logistic Regression — shows top features from BOTH Russian and English
print('⚙️ Computing SHAP values...')

lr_model = trained_models['Logistic Regression']
explainer = shap.LinearExplainer(lr_model, X_train_tfidf, feature_perturbation='interventional')
shap_values = explainer.shap_values(X_test_tfidf[:500])

feature_names   = tfidf.get_feature_names_out()
shap_importance = np.abs(shap_values).mean(axis=0)
top_idx      = np.argsort(shap_importance)[-25:][::-1]
top_features = [feature_names[i] for i in top_idx]
top_values   = shap_importance[top_idx]
coef_dir     = lr_model.coef_[0][top_idx]
bar_colors   = ['#e74c3c' if c > 0 else '#2ecc71' for c in coef_dir]

fig, ax = plt.subplots(figsize=(13, 9))
ax.barh(range(len(top_features)), top_values, color=bar_colors, edgecolor='black', alpha=0.85)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features, fontsize=11)
ax.set_xlabel('Mean |SHAP value| (impact on prediction)', fontsize=12)
ax.set_title(
    'Top-25 Features by SHAP Importance (RU + EN)\nRed = toxicity indicator | Green = neutrality indicator',
    fontsize=13, fontweight='bold'
)
red_p   = mpatches.Patch(color='#e74c3c', label='Toxicity feature')
green_p = mpatches.Patch(color='#2ecc71', label='Neutrality feature')
ax.legend(handles=[red_p, green_p], fontsize=11)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('shap_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart 11 saved: shap_feature_importance.png')

In [ ]:
print('\n' + '='*65)
print('✅ ALL DONE! FILES FOR DIPLOMA:')
print('='*65)
files = [
    'class_distribution.png          — Chart 1 (classes + language)',
    'text_length_distribution.png    — Chart 2 (lengths RU+EN)',
    'wordcloud.png                   — Chart 3 (4 word clouds)',
    'baseline_comparison.png         — Chart 4 (baseline metrics)',
    'confusion_matrices_baseline.png — Chart 5 (confusion matrices)',
    'roc_curves_baseline.png         — Chart 6 (ROC curves)',
    'bert_training_history.png       — Chart 7 (training history)',
    'bert_evaluation.png             — Chart 8 (BERT evaluation)',
    'final_comparison.png            — Chart 9 (all models)',
    'radar_chart.png                 — Chart 10 (radar)',
    'shap_feature_importance.png     — Chart 11 (SHAP)',
    'final_results.csv               — Final metrics table',
    'rubert_best.pt                  — Trained BERT weights',
]
for f in files:
    print(f'  📄 {f}')

print('\n📌 DATASETS USED:')
print(f'   RU: s-nlp/ru_paradetox (HuggingFace) — {len(df_ru_final)} examples')
print(f'   EN: Jigsaw Toxic Comments (Kaggle)   — {len(df_en_final)} examples')
print(f'   Total: {len(df_final)} examples')